# v11 — a correlacao surrogate&ndash;quantico replica entre corpora?

A afirmacao central do artigo e' que o kernel de fidelidade e' acompanhado por um surrogate
gaussiano classico. Na SVD isso deu **&rho; = 0,964** sobre 48 celulas de grade
(p = 5,6&times;10<sup>&minus;28</sup>). No Parkinson deu **&rho; = 0,363** (p = 0,096) sobre 22
celulas — nao significativo.

Duas leituras levam a artigos diferentes:

- **Ruido amostral.** Menos celulas, 195 gravacoes, IC largos. A tese sobrevive com ressalva.
- **Efeito real.** O acompanhamento na SVD depende de propriedades daquele corpus (79 features,
  52 delas de MFCC, fortemente redundantes) e nao e' fato geral sobre kernels de fidelidade.
  A tese vira condicional ao regime.

**O que este notebook faz:** roda os dois corpora na **mesma grade de 48 celulas**, repete o
experimento sobre varias particoes de folds, e produz uma **distribuicao de &rho; por corpus**
em vez de um numero unico. A comparacao e' entre distribuicoes.

A unidade de reamostragem e' a **particao de folds**, nao a celula: as 48 celulas compartilham
os mesmos folds e nao sao independentes — reamostrar celulas produziria IC falsamente estreito,
o mesmo erro de pseudorreplicacao ja documentado nos subgrupos.

In [ ]:
# !pip install pennylane -q
import numpy as np, pandas as pd, time, os
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, mannwhitneyu
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

def _sf(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            if ent:
                for i in range(q-1):
                    qml.CNOT(wires=[i, i+1])
                    qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                    qml.CNOT(wires=[i, i+1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _sf(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s+batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T)**2

def fit_surrogate(K, A):
    D2 = ((A[:, None, :]-A[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K-np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 8), method="bounded")
    return float(np.exp(r.x)), float(r.fun)/np.sqrt(K.size)

def subsample(ytr, gtr, npc, rng):
    idx = []
    for cls in (0, 1):
        take = []
        for s in rng.permutation(np.unique(gtr[ytr == cls])):
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= npc: break
        idx += take[:npc]
    return np.array(idx)

## 1. Corpora

Mesmos carregadores das rodadas anteriores. A auditoria do Parkinson conta os codigos de sujeito
extraidos do campo `name` — o arquivo traz 32, enquanto a documentacao da UCI descreve 31
pessoas. Reporte o numero medido e cite a discrepancia em nota.

In [ ]:
def carrega_svd(path="features_v5.pkl"):
    META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]
    f = pd.read_pickle(path)
    F = [x for x in f.columns if x not in META]
    return (f[F].values.astype(float), f.label.values.astype(int),
            f.speaker_id.values, "SVD")

def carrega_parkinsons(path="parkinsons.data"):
    d = pd.read_csv(path)
    subj = d.name.str.extract(r'(phon_R\d+_S\d+)')[0].values
    F = [c for c in d.columns if c not in ("name", "status")]
    return d[F].values.astype(float), d.status.values.astype(int), subj, "Parkinson"

CORPORA = []
for fn, npc, folds in [(carrega_parkinsons, None, 4), (carrega_svd, 200, 10)]:
    try:
        X, y, g, nome = fn()
        CORPORA.append(dict(nome=nome, X=X, y=y, g=g, npc=npc, folds=folds))
    except FileNotFoundError:
        print(f"ausente: {fn.__name__} — sera ignorado")

painel("corpora carregados",
       [f"<b>{c['nome']}</b>: {c['X'].shape[0]} amostras &middot; {c['X'].shape[1]} features "
        f"&middot; {pd.unique(c['g']).size} grupos &middot; prev={c['y'].mean():.3f} "
        f"&middot; {c['folds']} folds" for c in CORPORA],
       destaque="Se a SVD nao aparecer, faca upload de <b>features_v5.pkl</b>. O notebook roda "
                "com o que estiver disponivel, mas a comparacao exige os dois.")

## 2. Grade unica

**48 celulas para os dois corpora** — e' o que torna os &rho; comparaveis. A grade anterior do
Parkinson tinha 22 celulas (grade da v7/v8, adensada em torno de c=1), e comparar &rho; sobre
numeros diferentes de celulas confunde efeito com resolucao.

In [ ]:
QUBITS = [4, 6, 8, 10]
SCALES = np.round(np.geomspace(0.01, 2.0, 12), 4)     # grade original da SVD
N_PARTICOES = 8
REPS_CIRCUITO = 2

n_ev = sum(len(QUBITS)*len(SCALES)*c["folds"]*N_PARTICOES for c in CORPORA)
painel("grade e orcamento",
       [f"{len(QUBITS)} qubits &times; {len(SCALES)} bandwidths = "
        f"<b>{len(QUBITS)*len(SCALES)} celulas</b> por corpus",
        f"{N_PARTICOES} particoes de folds por corpus",
        f"&asymp; {n_ev} avaliacoes de kernel no total",
        f"estimativa: {n_ev*0.9/60:.0f}-{n_ev*1.7/60:.0f} min em CPU"],
       destaque="A particao e' a unidade de reamostragem. Aumentar <b>N_PARTICOES</b> estreita "
                "a distribuicao de &rho;; aumentar celulas nao — celulas compartilham folds.")

In [ ]:
def rho_de_uma_particao(X, y, g, folds, npc, seed):
    """Roda a grade inteira numa particao e devolve rho(auc_q, auc_surr) sobre as celulas."""
    rows = []
    sg = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(sg.split(X, y, g)):
        rng = np.random.default_rng(seed*100 + fold)
        idx = subsample(y[tr], g[tr], npc, rng) if npc else np.arange(len(tr))
        ytr, yte = y[tr][idx], y[te]
        imp = SimpleImputer(strategy="median").fit(X[tr][idx])
        Xtr, Xte = imp.transform(X[tr][idx]), imp.transform(X[te])
        for q in QUBITS:
            k = min(q, Xtr.shape[1])
            sel = SelectKBest(f_classif, k=k).fit(Xtr, ytr)
            sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
            A0 = sc.transform(sel.transform(Xtr))
            B0 = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
            for cv in SCALES:
                A, B = A0*cv, B0*cv
                S1, S2 = states(A, k, REPS_CIRCUITO), states(B, k, REPS_CIRCUITO)
                K = gram_fid(S1, S1)
                m = SVC(kernel="precomputed").fit(K, ytr)
                gs, resid = fit_surrogate(K, A)
                ms = SVC(kernel="rbf", gamma=gs).fit(A, ytr)
                rows.append(dict(q=q, c=cv,
                                 aq=roc_auc_score(yte, m.decision_function(gram_fid(S2, S1))),
                                 asr=roc_auc_score(yte, ms.decision_function(B)),
                                 resid=resid))
    cel = pd.DataFrame(rows).groupby(["q", "c"])[["aq", "asr", "resid"]].mean().reset_index()
    r, p = spearmanr(cel.aq, cel.asr)
    return r, p, cel

resultados, t0 = {}, time.time()
for cp in CORPORA:
    rs = []
    for s in range(N_PARTICOES):
        r, p, cel = rho_de_uma_particao(cp["X"], cp["y"], cp["g"], cp["folds"],
                                        cp["npc"], seed=7 + s)
        rs.append(dict(particao=s, rho=round(r, 4), p=p,
                       dif_mediana=round((cel.aq - cel.asr).abs().median(), 4),
                       auc_max=round(cel.aq.max(), 4)))
        print(f"{cp['nome']} particao {s} — rho={r:+.3f} — {time.time()-t0:.0f}s")
    resultados[cp["nome"]] = pd.DataFrame(rs)

resumo = pd.DataFrame([
    dict(corpus=n, particoes=len(d), rho_mediano=round(d.rho.median(), 3),
         rho_min=round(d.rho.min(), 3), rho_max=round(d.rho.max(), 3),
         dif_mediana=round(d.dif_mediana.median(), 4))
    for n, d in resultados.items()])
painel("distribuicao de rho por corpus",
       [f"{len(QUBITS)*len(SCALES)} celulas por particao, {N_PARTICOES} particoes",
        "dif_mediana = |AUC quantico &minus; AUC surrogate| mediano sobre as celulas"],
       tabela=resumo)

## 3. As duas distribuicoes se sobrepoem?

Se as faixas de &rho; se sobrepuserem, a diferenca entre 0,964 e 0,363 era resolucao amostral e
a tese do artigo sobrevive. Se nao se sobrepuserem, o acompanhamento pelo surrogate **depende do
corpus**, e a afirmacao tem de ser reescrita como condicional ao regime.

In [ ]:
if len(resultados) >= 2:
    (n1, d1), (n2, d2) = list(resultados.items())[:2]
    sobrepoe = not (d1.rho.max() < d2.rho.min() or d2.rho.max() < d1.rho.min())
    u, pmw = mannwhitneyu(d1.rho, d2.rho)
    dif = d2.rho.median() - d1.rho.median()

    if sobrepoe:
        dest = ("As faixas <b>se sobrepoem</b>: a diferenca entre os &rho; publicados era "
                "resolucao amostral. A tese do artigo sobrevive — reporte a faixa entre "
                "corpora, nao o valor de um deles.")
    elif pmw < 0.05:
        dest = (f"As faixas <b>nao se sobrepoem</b> (Mann-Whitney p={pmw:.3g}). O "
                "acompanhamento pelo surrogate depende do corpus. A afirmacao central tem de "
                "virar condicional: <i>em corpora de alta dimensionalidade e features "
                "redundantes, o kernel de fidelidade e' reproduzido por um surrogate "
                "gaussiano</i> — e o Parkinson passa a ser o contraexemplo que delimita o "
                "escopo, o que e' mais informativo que uma generalizacao fragil.")
    else:
        dest = (f"Faixas disjuntas mas sem significancia (p={pmw:.3g}) com {N_PARTICOES} "
                "particoes. Aumente N_PARTICOES antes de concluir.")

    painel("comparacao entre corpora",
           [f"{n1}: mediana {d1.rho.median():+.3f} "
            f"[{d1.rho.min():+.3f}, {d1.rho.max():+.3f}]",
            f"{n2}: mediana {d2.rho.median():+.3f} "
            f"[{d2.rho.min():+.3f}, {d2.rho.max():+.3f}]",
            f"diferenca das medianas = <b>{dif:+.3f}</b> &nbsp;|&nbsp; "
            f"Mann-Whitney p = {pmw:.3g}",
            f"faixas se sobrepoem: <b>{sobrepoe}</b>"],
           tabela=pd.concat([d.assign(corpus=n) for n, d in resultados.items()])
                   [["corpus", "particao", "rho", "dif_mediana", "auc_max"]],
           destaque=dest)
else:
    painel("comparacao entre corpora", ["apenas um corpus disponivel"],
           destaque="Faca upload do corpus que falta para executar a comparacao.")

## 4. O que reportar

- **Se sobrepuserem:** a Introducao segue como esta; Resultados reporta a faixa de &rho; entre
  corpora e o numero de particoes.
- **Se nao sobrepuserem:** a afirmacao central vira condicional ao regime, e o Parkinson entra
  como contraexemplo que delimita o escopo. Um escopo delimitado por evidencia e' mais forte,
  numa revisao, que uma generalizacao que o proprio material contradiz.
- Em qualquer caso: nota de rodape sobre os 32 codigos de sujeito do arquivo do Parkinson contra
  as 31 pessoas descritas pela UCI.